# Time to 3D rendering (skimage version)
Converts the video from an Echography to a 3D rendering where time becomes the 3D dimension.

Uses **scikit-image Marching Cubes** algorithm (replaces VTK dependency).
Renders the interactive 3D view with **Plotly**.

Dependencies: `pip install scikit-image opencv-python plotly numpy`

In [ ]:
import numpy as np
import cv2
from skimage.measure import marching_cubes

# ── Configuration ──────────────────────────────────────────────────────────────
VIDEOFILE   = "./Anonymized_EchoCardiography06.mp4"
ISO_LEVEL   = 127      # grayscale threshold (same as VTK original)
STEP_SIZE   = 1        # marching-cubes step size (increase to 2-3 for faster/coarser)
MAX_FRAMES  = None     # set to an int (e.g. 60) to limit memory usage

# ── Read video ─────────────────────────────────────────────────────────────────
print(f"Reading {VIDEOFILE} ...")
video = cv2.VideoCapture(VIDEOFILE)
print("Video opened:", video.isOpened())

frames = []
while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frames.append(gray)
    if MAX_FRAMES and len(frames) >= MAX_FRAMES:
        break

video.release()
print(f"Frames read: {len(frames)}")

# Stack: shape = (n_frames, height, width)  →  axes = (T, Y, X)
volume = np.stack(frames).astype(np.uint8)
print(f"Volume shape (T, H, W): {volume.shape}")

In [ ]:
# ── Marching Cubes (skimage) ───────────────────────────────────────────────────
# Returns:
#   verts  – (N, 3) float array of vertex coordinates  (T, Y, X)
#   faces  – (M, 3) int   array of triangle vertex indices
#   normals, values – per-vertex normals and iso-values
print(f"Running Marching Cubes (iso={ISO_LEVEL}, step={STEP_SIZE}) ...")
verts, faces, normals, values = marching_cubes(
    volume,
    level=ISO_LEVEL,
    step_size=STEP_SIZE,
    allow_degenerate=False,
)
print(f"Vertices : {verts.shape}")
print(f"Faces    : {faces.shape}")

In [ ]:
import plotly.graph_objects as go

# verts columns: 0=T (frame/time), 1=Y (height), 2=X (width)
t_coord = verts[:, 0]                              # X axis → "Echography Frame #"
x_coord = verts[:, 2]                              # Y axis → "Width"
y_coord = verts[:, 1].max() - verts[:, 1]          # Z axis → "Height" (flipped)

# Vertex coloring: intensity along time axis
intensity = t_coord / t_coord.max()

fig = go.Figure(data=[
    go.Mesh3d(
        x=t_coord,
        y=x_coord,
        z=y_coord,
        colorbar_title='Frame #',
        colorscale=[
            [0.0, 'gold'],
            [0.2, 'mediumturquoise'],
            [1.0, 'magenta'],
        ],
        intensity=intensity,
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        name='echography surface',
        showscale=True,
        flatshading=False,
    )
])

fig.update_layout(
    title=dict(text="Time to 3D — Use mouse to rotate, zoom, etc."),
    scene=dict(
        xaxis=dict(title='Echography Frame #'),
        yaxis=dict(title='Width (px)'),
        zaxis=dict(title='Height (px)'),
        # 'manual' + computed ratios → all three axes appear the same physical length
        aspectmode='manual',
        aspectratio=dict(x=ar_t, y=ar_x, z=ar_y),
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)

# Save and show
OUTPUT_HTML = "/mnt/d/html/vtk3d_skimage.html"   # adjust path as needed
fig.write_html(OUTPUT_HTML)
print(f"Saved to {OUTPUT_HTML}")
fig.show()